Semana 1: carga y reporte de telecom-churn-dataset
En este notebook se construye un flujo completo de trabajo:

Configurar la ruta del proyecto para importar las clases reutilizables.
Cargar Fast_Food_Restaurants desde Hugging Face.
Obtener un DataFrame de pandas y revisar su estructura.
Generar un reporte en pantalla y guardarlo en un archivo .txt.
Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con hf auth login.

1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
    
sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/PI3_Analitica_EA1/PI3_Analitica_EA1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Configurar el dataset y la salida
dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/sincronicas/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="Zainab-Jamil30/telecom-churn-dataset",
    split="train",
    filename="datos/telco_churn.csv",
)

3. Descargar el dataset y crear el DataFrame
El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head(10)

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 10000/10000 [00:00<00:00, 359329.03 examples/s]

,CustomerID,Gender,Age,Tenure,ContractType,InternetService,MonthlyCharges,TotalCharges,PaymentMethod,TechSupport,StreamingTV,OnlineSecurity,PaperlessBilling,Churn
0,CUST00001,Male,68,16,Month-to-month,Fiber optic,63.46,1033.67,Electronic check,Yes,Yes,No,No,1
1,CUST00002,Female,57,51,Month-to-month,Fiber optic,79.29,4064.48,Mailed check,Yes,Yes,No,Yes,1
2,CUST00003,Male,24,11,Month-to-month,DSL,69.53,748.64,Bank transfer,Yes,No,No,Yes,1
3,CUST00004,Male,49,47,Month-to-month,Fiber optic,95.02,4527.19,Electronic check,Yes,No,No,Yes,1
4,CUST00005,Male,65,21,Month-to-month,Fiber optic,59.70,1272.96,Electronic check,No,Yes,Yes,Yes,1
5,CUST00006,Female,36,48,Month-to-month,DSL,72.66,3481.83,Electronic check,No,No,No,Yes,0
6,CUST00007,Male,23,51,Month-to-month,DSL,45.04,2344.69,Mailed check,Yes,No,No,No,0
7,CUST00008,Male,37,0,Month-to-month,Fiber optic,57.19,8.35,Mailed check,No,Yes,No,Yes,1
8,CUST00009,Male,44,65,Two year,DSL,52.57,3432.30,Electronic check,No,No,No,Yes,0
9,CUST00010,Female,70,63,Month-to-month,DSL,48.18,2951.96,Mailed check,No,No,Yes,Yes,0


4. Revisar la estructura del DataFrame
Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 10000
Columnas: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        10000 non-null  object 
 1   Gender            10000 non-null  object 
 2   Age               10000 non-null  int64  
 3   Tenure            10000 non-null  int64  
 4   ContractType      10000 non-null  object 
 5   InternetService   10000 non-null  object 
 6   MonthlyCharges    10000 non-null  float64
 7   TotalCharges      10000 non-null  float64
 8   PaymentMethod     10000 non-null  object 
 9   TechSupport       10000 non-null  object 
 10  StreamingTV       10000 non-null  object 
 11  OnlineSecurity    10000 non-null  object 
 12  PaperlessBilling  10000 non-null  object 
 13  Churn             10000 non-null  int64  
dtypes: float64(2), int64(3), object(9)
memory usage: 1.1+ MB


6. transformaciones basicas 

In [5]:
import pandas as pd

# esta transformacion genera un segmento según la fidelidad del usuario
df["CategoriaTenure"] = pd.cut(
    df["Tenure"],
    bins=[0, 12, 24, 48, 100],
    labels=[
        "Nuevo",
        "Estable",
        "Fiel",
        "Muy Fiel"
    ],
    include_lowest=True
)
# se crea una columna con un segmento de edad
df["GrupoEdad"] = pd.cut(
    df["Age"],
    bins=[0, 30, 50, 65, 100],
    labels=["Joven", "Adulto", "Adulto Maduro", "Senior"]
)
# para las columnas que tienen un yes o no se reemplaza por valores de 0 y 1, para posteriores analisis de segmentos
columnas = [
    "TechSupport",
    "StreamingTV",
    "OnlineSecurity",
    "PaperlessBilling"
]
for col in columnas:
    df[col] = df[col].map({
        "Yes": 1,
        "No": 0
    })
# se genera una nueva columna que suma el total de servicios y tener el valor para crear segmentos
df["TotalServicios"] = (
    df["TechSupport"] +
    df["StreamingTV"] +
    df["OnlineSecurity"]
)
# se genera una nueva columna que consolida los servicios contratados
df["ServiciosContratados"] = (
    df[["TechSupport", "StreamingTV", "OnlineSecurity"]]
    .apply(
        lambda fila: ", ".join(fila.index[fila == 1]),
        axis=1
    )
)

df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   CustomerID            10000 non-null  object  
 1   Gender                10000 non-null  object  
 2   Age                   10000 non-null  int64   
 3   Tenure                10000 non-null  int64   
 4   ContractType          10000 non-null  object  
 5   InternetService       10000 non-null  object  
 6   MonthlyCharges        10000 non-null  float64 
 7   TotalCharges          10000 non-null  float64 
 8   PaymentMethod         10000 non-null  object  
 9   TechSupport           10000 non-null  int64   
 10  StreamingTV           10000 non-null  int64   
 11  OnlineSecurity        10000 non-null  int64   
 12  PaperlessBilling      10000 non-null  int64   
 13  Churn                 10000 non-null  int64   
 14  CategoriaTenure       10000 non-null  category
 15  Gru

,CustomerID,Gender,Age,Tenure,ContractType,InternetService,MonthlyCharges,TotalCharges,PaymentMethod,TechSupport,StreamingTV,OnlineSecurity,PaperlessBilling,Churn,CategoriaTenure,GrupoEdad,TotalServicios,ServiciosContratados
0,CUST00001,Male,68,16,Month-to-month,Fiber optic,63.46,1033.67,Electronic check,1,1,0,0,1,Estable,Senior,2,"TechSupport, StreamingTV"
1,CUST00002,Female,57,51,Month-to-month,Fiber optic,79.29,4064.48,Mailed check,1,1,0,1,1,Muy Fiel,Adulto Maduro,2,"TechSupport, StreamingTV"
2,CUST00003,Male,24,11,Month-to-month,DSL,69.53,748.64,Bank transfer,1,0,0,1,1,Nuevo,Joven,1,TechSupport
3,CUST00004,Male,49,47,Month-to-month,Fiber optic,95.02,4527.19,Electronic check,1,0,0,1,1,Fiel,Adulto,1,TechSupport
4,CUST00005,Male,65,21,Month-to-month,Fiber optic,59.70,1272.96,Electronic check,0,1,1,1,1,Estable,Adulto Maduro,2,"StreamingTV, OnlineSecurity"
5,CUST00006,Female,36,48,Month-to-month,DSL,72.66,3481.83,Electronic check,0,0,0,1,0,Fiel,Adulto,0,
6,CUST00007,Male,23,51,Month-to-month,DSL,45.04,2344.69,Mailed check,1,0,0,0,0,Muy Fiel,Joven,1,TechSupport
7,CUST00008,Male,37,0,Month-to-month,Fiber optic,57.19,8.35,Mailed check,0,1,0,1,1,Nuevo,Adulto,1,StreamingTV
8,CUST00009,Male,44,65,Two year,DSL,52.57,3432.30,Electronic check,0,0,0,1,0,Muy Fiel,Adulto,0,
9,CUST00010,Female,70,63,Month-to-month,DSL,48.18,2951.96,Mailed check,0,0,1,1,0,Muy Fiel,Senior,1,OnlineSecurity


5. Importar la clase de reportes
Reportes recibe un DataFrame en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo .txt.

In [6]:
from clases_po import Reportes

6. Generar y visualizar el reporte como texto
generar_reporte(df) devuelve un str con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en texto y se imprime.

In [7]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 18
Duplicados: 0

Columnas y tipos:
- CustomerID: object
- Gender: object
- Age: int64
- Tenure: int64
- ContractType: object
- InternetService: object
- MonthlyCharges: float64
- TotalCharges: float64
- PaymentMethod: object
- TechSupport: int64
- StreamingTV: int64
- OnlineSecurity: int64
- PaperlessBilling: int64
- Churn: int64
- CategoriaTenure: category
- GrupoEdad: category
- TotalServicios: int64
- ServiciosContratados: object

Valores nulos por columna:
- CustomerID: 0
- Gender: 0
- Age: 0
- Tenure: 0
- ContractType: 0
- InternetService: 0
- MonthlyCharges: 0
- TotalCharges: 0
- PaymentMethod: 0
- TechSupport: 0
- StreamingTV: 0
- OnlineSecurity: 0
- PaperlessBilling: 0
- Churn: 0
- CategoriaTenure: 0
- GrupoEdad: 0
- TotalServicios: 0
- ServiciosContratados: 0

Estadísticas numéricas:
             Age     Tenure  MonthlyCharges  TotalCharges  TechSupport  StreamingTV  OnlineSecurity  PaperlessBilling      Churn  TotalServicios
count

7. Imprimir el reporte directamente
imprimir_reporte(df) ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [8]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 18
Duplicados: 0

Columnas y tipos:
- CustomerID: object
- Gender: object
- Age: int64
- Tenure: int64
- ContractType: object
- InternetService: object
- MonthlyCharges: float64
- TotalCharges: float64
- PaymentMethod: object
- TechSupport: int64
- StreamingTV: int64
- OnlineSecurity: int64
- PaperlessBilling: int64
- Churn: int64
- CategoriaTenure: category
- GrupoEdad: category
- TotalServicios: int64
- ServiciosContratados: object

Valores nulos por columna:
- CustomerID: 0
- Gender: 0
- Age: 0
- Tenure: 0
- ContractType: 0
- InternetService: 0
- MonthlyCharges: 0
- TotalCharges: 0
- PaymentMethod: 0
- TechSupport: 0
- StreamingTV: 0
- OnlineSecurity: 0
- PaperlessBilling: 0
- Churn: 0
- CategoriaTenure: 0
- GrupoEdad: 0
- TotalServicios: 0
- ServiciosContratados: 0

Estadísticas numéricas:
             Age     Tenure  MonthlyCharges  TotalCharges  TechSupport  StreamingTV  OnlineSecurity  PaperlessBilling      Churn  TotalServicios
count

8. Guardar el reporte en un archivo de texto
guardar_reporte(df, ruta) crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en src/sincronicas/reportes/fast_food_restaurants.txt cuando se ejecute desde src/sincronicas.

In [9]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/telco_churn.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/telco_churn.txt
